In [2]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GroqApi")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("Sri Sai Clinic Appointments") # run once; next time use gc.open("Sri Sai Clinic Appointments")
ws = sh.sheet1
ws.append_row(["patient_name", "date", "time", "fee"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/12TVLiSAvp5BtM-5hBeaApr3VdiUfswwe3mW8sEa1F4U


In [4]:
from pydantic import BaseModel, Field
from langchain.tools import tool

CONSULTATION_FEE = 300
MAX_SLOTS_PER_HOUR = 6

HOURS = {
    "monday": "9:00 AM - 1:00 PM and 5:00 PM - 8:00 PM",
    "tuesday": "9:00 AM - 1:00 PM and 5:00 PM - 8:00 PM",
    "wednesday": "9:00 AM - 1:00 PM and 5:00 PM - 8:00 PM",
    "thursday": "9:00 AM - 1:00 PM and 5:00 PM - 8:00 PM",
    "friday": "9:00 AM - 1:00 PM and 5:00 PM - 8:00 PM",
    "saturday": "9:00 AM - 1:00 PM and 5:00 PM - 8:00 PM",
    "sunday": "Closed"
}

class Appointment(BaseModel):
    """A patient appointment taken from a chat message."""
    name: str = Field(description="Patient's full or first name")
    date: str = Field(description="Appointment date as the patient said it")
    time: str = Field(description="Preferred appointment time")

@tool
def get_consultation_fee(num_appointments: int = 1) -> str:
    """Get the total consultation fee in rupees for one or more appointments."""
    return f"{num_appointments} appointment(s) = Rs {CONSULTATION_FEE * num_appointments}"

@tool
def check_clinic_hours(day: str) -> str:
    """Check Sri Sai Clinic's opening hours for a day of the week, e.g. 'sunday'."""
    hours = HOURS.get(day.lower().strip())
    if not hours:
        return f"'{day}' is not a valid day of the week."
    if hours == "Closed":
        return f"The clinic is closed on {day.capitalize()}."
    return f"Open {hours} on {day.capitalize()}."

@tool
def book_appointment(name: str, date: str, time: str) -> str:
    """Save a confirmed appointment to the Appointments sheet.
    Use ONLY after collecting name, date, and time AND confirming them with the patient."""
    try:
        appt = Appointment(name=name, date=date, time=time)
    except Exception as e:
        return f"Could not save the appointment: {e}. Ask the patient to correct it."

    # Check hourly slot capacity (max 6 per hour, 10 mins each)
    existing_records = ws.get_all_records()
    hour_prefix = appt.time.split(":")[0].strip().lower()
    booked_in_hour = sum(
        1 for row in existing_records
        if str(row.get("date", "")).lower().strip() == appt.date.lower().strip()
        and str(row.get("time", "")).lower().strip().startswith(hour_prefix)
    )
    if booked_in_hour >= MAX_SLOTS_PER_HOUR:
        return f"Sorry, all 6 slots for that hour on {appt.date} are full. Please suggest another time."

    ws.append_row([appt.name, appt.date, appt.time, CONSULTATION_FEE])
    return f"Booked: 10-minute slot for {appt.name} on {appt.date} at {appt.time}. Fee: Rs {CONSULTATION_FEE}."

In [6]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the receptionist for Sri Sai Clinic. Your job is to answer basic clinic questions and book patient appointments.
FACTS:  Clinic Name: Sri Sai Clinic. Doctor: Dr. Ramesh (General Physician).
        Each appointment is 10 minutes (maximum 6 appointments per hour).
RULES:  - Always use get_consultation_fee for fee questions and check_clinic_hours for timings; never guess them.
        - You must collect three details to book an appointment: patient's name, date, and time.
        - Repeat and confirm all three details (name, date, and time) with the caller BEFORE calling book_appointment.
        - Never give medical advice. For any medical questions, state that Dr. Ramesh can discuss it during the appointment, or offer to connect them to reception staff.
FORMAT: Reply clearly in under 50 simple words."""

agent = create_agent(
    model=llm,
    tools=[get_consultation_fee, check_clinic_hours, book_appointment],
    system_prompt=SYSTEM
)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

# Test conversation
print(chat("Hi, I'm Akshay. Are you open on Saturday? How much is the consultation fee?"))
print(chat("Okay, book me for tomorrow at 1:30 PM."))
print(chat("Yes, those details are correct. Please confirm."))

Hi Akshay! We’re open Saturday from 9 AM‑1 PM and 5 PM‑8 PM. One 10‑minute consultation costs Rs 300. Let me know if you’d like to book an appointment.
Sure! Just to confirm:

- Name: Akshay  
- Date: tomorrow  
- Time: 1:30 PM  

Is that correct? If so, I’ll book it for you.
All set! Your 10‑minute slot with Dr. Ramesh is booked for tomorrow at 1:30 PM. The fee is Rs 300. See you then!


In [ ]:
for m in history:
    m.pretty_print()

================================ Human Message =================================

Hi, I'm Ravi. Are you open on Sunday? How much is the consultation fee?
================================== Ai Message ==================================
Tool Calls:
  check_clinic_hours (fc_ae968a0c-a3f4-496d-afe1-47fa4b6b6303)
 Call ID: fc_ae968a0c-a3f4-496d-afe1-47fa4b6b6303
  Args:
    day: sunday
================================= Tool Message =================================
Name: check_clinic_hours

The clinic is closed on Sunday.
================================== Ai Message ==================================
Tool Calls:
  get_consultation_fee (fc_22de7eb1-f636-4cc5-add3-d6a0d38d64a9)
 Call ID: fc_22de7eb1-f636-4cc5-add3-d6a0d38d64a9
  Args:
    num_appointments: 1
================================= Tool Message =================================
Name: get_consultation_fee

1 appointment(s) = Rs 300
================================== Ai Message ==================================

The clinic is closed

In [3]:
from typing import Optional
from pydantic import BaseModel, Field

# 1. Define a richer Pydantic record for Sri Sai Clinic
class ClinicRecord(BaseModel):
    patient_name: str = Field(description="Patient's full or first name")
    date: str = Field(description="Requested appointment date or day, e.g., 'tomorrow', 'Monday', '2026-11-30'")
    time: Optional[str] = Field(default=None, description="Preferred time or time of day, e.g., '10:30 AM', 'evening'")
    symptoms: Optional[str] = Field(default=None, description="Reason for visit or health issue mentioned")
    is_urgent: bool = Field(default=False, description="True if the patient mentions pain, high fever, or urgency")

# 2. Attach it to your existing `llm`
structured_clinic_llm = llm.with_structured_output(ClinicRecord)

# 3. Test on 5 messy patient messages
messy_patient_messages = [
    "hi doctor dis is Suresh, need 2 come tmrw around 6pm for severe toothache n fever",
    "Plz book slot for Mrs. Lakshmi on Monday morning 10:30 for routine BP check",
    "hey can u fix an appt? my name is Priya, coming Friday evening",
    "Yo! Bad stomach pain since morning, Ankit here. Can Dr Ramesh see me today at 5?",
    "Need general checkup next wednesday, name Rahul"
]

# 4. Run and inspect the structured output
for i, msg in enumerate(messy_patient_messages, 1):
    record = structured_clinic_llm.invoke(msg)
    print(f"--- Message {i} ---")
    print("Raw:   ", msg)
    print("Parsed:", record)
    print()

--- Message 1 ---
Raw:    hi doctor dis is Suresh, need 2 come tmrw around 6pm for severe toothache n fever
Parsed: patient_name='Suresh' date='2026-11-28' time='6pm' symptoms='severe toothache and fever' is_urgent=True

--- Message 2 ---
Raw:    Plz book slot for Mrs. Lakshmi on Monday morning 10:30 for routine BP check
Parsed: patient_name='Mrs. Lakshmi' date='Monday' time='10:30' symptoms='routine BP check' is_urgent=False

--- Message 3 ---
Raw:    hey can u fix an appt? my name is Priya, coming Friday evening
Parsed: patient_name='Priya' date='Friday evening' time=None symptoms=None is_urgent=False

--- Message 4 ---
Raw:    Yo! Bad stomach pain since morning, Ankit here. Can Dr Ramesh see me today at 5?
Parsed: patient_name='Ankit' date='2026-11-27' time='5' symptoms='stomach pain' is_urgent=True

--- Message 5 ---
Raw:    Need general checkup next wednesday, name Rahul
Parsed: patient_name='Rahul' date='next wednesday' time=None symptoms=None is_urgent=False

